# PhoBERT Fine-tuning for Sentiment Analysis (UIT-VSFC)

#1. Cài đặt thư viện

In [ ]:
!pip install transformers datasets torch scikit-learn

# 2. Import thư viện

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 3. Load dữ liệu (train / dev / test)

In [ ]:
def load_vsfc_data(path):
    with open(f"{path}/sents.txt", encoding="utf-8") as f:
        texts = [line.strip() for line in f]

    with open(f"{path}/sentiments.txt") as f:
        labels = [int(line.strip()) for line in f]

    return texts, labels

train_texts, train_labels = load_vsfc_data("./train")
dev_texts, dev_labels = load_vsfc_data("./dev")
test_texts, test_labels = load_vsfc_data("./test")

print(len(train_texts))
print(len(dev_texts))
print(len(test_texts))

11426
1583
3166


# 4. Tokenizer

In [ ]:
model_name = "vinai/phobert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
sample = train_texts[0]
print(sample)

encoded = tokenizer(sample, max_length=256, truncation=True, padding="max_length")
print(encoded)

slide giáo trình đầy đủ .
{'input_ids': [0, 48090, 4368, 1893, 545, 312, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0


# 5. Dataset

In [ ]:
class VSFCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len
        )
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = VSFCDataset(train_texts, train_labels, tokenizer)
dev_dataset = VSFCDataset(dev_texts, dev_labels, tokenizer)
test_dataset = VSFCDataset(test_texts, test_labels, tokenizer)

print(train_dataset[0])

{'input_ids': tensor([    0, 48090,  4368,  1893,   545,   312,     5,     2,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
            1,     1,     1,     1,     1,     1, 

# 6. Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

# 7. Metric

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

# 8. Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    # --- Optimization ---
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,

    # --- Evaluation & Saving ---
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    # --- Logging ---
    logging_strategy="steps",
    logging_steps=10,

    # --- Best model ---
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # --- Performance ---
    fp16=torch.cuda.is_available(),   # tự bật nếu có GPU
    dataloader_num_workers=2,

    # --- Reproducibility ---
    seed=42
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.109424,0.233539,0.938724,0.811609
2,0.078430,0.223907,0.942514,0.842769
3,0.116418,0.254214,0.941251,0.817276


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=2145, training_loss=0.20256625113787352, metrics={'train_runtime': 454.1964, 'train_samples_per_second': 75.47, 'train_steps_per_second': 4.723, 'total_flos': 2871283754803716.0, 'train_loss': 0.20256625113787352, 'epoch': 3.0})

# 9. Evaluate (test set)

In [ ]:
results = trainer.evaluate(test_dataset)
print(results)

{'eval_loss': 0.2662109136581421, 'eval_accuracy': 0.9295641187618446, 'eval_f1': 0.8156110501912499, 'eval_runtime': 5.0359, 'eval_samples_per_second': 628.686, 'eval_steps_per_second': 39.318, 'epoch': 3.0}


# 10. Inference

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def predict(text):
    model.to(device)   # đảm bảo model ở đúng device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():   # không cần gradient khi inference
        outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred = torch.argmax(probs, dim=1).item()

    return pred

In [ ]:
predict("Giảng viên dạy rất hay")

2

In [ ]:
predict("Buồn ngủ vô cùng")

0

# 11. Đánh giá trên tập test

In [ ]:
# Lấy prediction từ Trainer

predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

In [ ]:
# Convert logits → label

import numpy as np

preds = np.argmax(logits, axis=1)

print(preds[:10])
print(labels[:10])

[2 2 2 0 0 2 1 2 0 2]
[2 2 2 2 0 2 2 2 0 2]


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    labels,
    preds,
    target_names=["negative", "neutral", "positive"],
    digits=4
))

              precision    recall  f1-score   support

    negative     0.9438    0.9532    0.9484      1409
     neutral     0.6148    0.4970    0.5497       167
    positive     0.9434    0.9541    0.9487      1590

    accuracy                         0.9296      3166
   macro avg     0.8340    0.8014    0.8156      3166
weighted avg     0.9262    0.9296    0.9275      3166



In [ ]:
import torch
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [ ]:
# 1. Cài đặt peft nếu chưa có
!pip install -q peft

In [ ]:
# 2. Khởi tạo model gốc
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

In [ ]:
# 3. Cấu hình LoRA
# Với PhoBERT (RoBERTa), các target_modules thường là query, value trong attention layers
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS # Sequence Classification
)

In [ ]:
# 4. Wrap model với LoRA
lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 887,811 || all params: 135,888,390 || trainable%: 0.6533


In [ ]:
# 5. Thiết lập TrainingArguments mới cho LoRA
lora_training_args = TrainingArguments(
    output_dir="./results_lora",
    learning_rate=1e-4, # LoRA thường cần LR cao hơn fine-tuning toàn bộ
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    remove_unused_columns=False # Quan trọng khi dùng PEFT
)

In [ ]:
# 6. Khởi tạo Trainer mới
lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
# 7. Huấn luyện
lora_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.145921,0.272319,0.914719,0.664592
2,0.147645,0.223983,0.934934,0.808872
3,0.179820,0.229133,0.933039,0.794821


TrainOutput(global_step=2145, training_loss=0.26112596666340504, metrics={'train_runtime': 275.8611, 'train_samples_per_second': 124.258, 'train_steps_per_second': 7.776, 'total_flos': 2901046627781640.0, 'train_loss': 0.26112596666340504, 'epoch': 3.0})

In [ ]:
# 8. Đánh giá trên tập test
lora_results = lora_trainer.evaluate(test_dataset)
print("LoRA Results:", lora_results)

LoRA Results: {'eval_loss': 0.27066996693611145, 'eval_accuracy': 0.9175615919140871, 'eval_f1': 0.777563530279259, 'eval_runtime': 5.3676, 'eval_samples_per_second': 589.833, 'eval_steps_per_second': 36.888, 'epoch': 3.0}


In [ ]:
import torch.nn as nn
from transformers import RobertaPreTrainedModel, RobertaModel, AutoModelForSequenceClassification

# 1. Load nhãn Topic
def load_topics(path):
    with open(f"{path}/topics.txt") as f:
        return [int(line.strip()) for line in f]

train_topics = load_topics("./train")
dev_topics = load_topics("./dev")
test_topics = load_topics("./test")

# 2. Định nghĩa kiến trúc Multi-task
class PhoBertMultiTask(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.roberta = RobertaModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

        # Head 1: Sentiment (3 nhãn)
        self.sentiment_classifier = nn.Linear(config.hidden_size, 3)

        # Head 2: Topic (4 nhãn)
        self.topic_classifier = nn.Linear(config.hidden_size, 4)

        # Khởi tạo trọng số
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, labels_sentiment=None, labels_topic=None):
        outputs = self.roberta(input_ids, attention_mask=attention_mask)
        pooled_output = outputs[1] # CLS token representation
        pooled_output = self.dropout(pooled_output)

        logits_sentiment = self.sentiment_classifier(pooled_output)
        logits_topic = self.topic_classifier(pooled_output)

        total_loss = None
        if labels_sentiment is not None and labels_topic is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss_s = loss_fct(logits_sentiment, labels_sentiment)
            loss_t = loss_fct(logits_topic, labels_topic)
            total_loss = loss_s + loss_t

        return {
            'loss': total_loss,
            'logits_sentiment': logits_sentiment,
            'logits_topic': logits_topic
        }

# Khởi tạo model
config = AutoModelForSequenceClassification.from_pretrained(model_name).config
multi_task_model = PhoBertMultiTask.from_pretrained(model_name, config=config)
print("Multi-task model initialized successfully.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

PhoBertMultiTask LOAD REPORT from: vinai/phobert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
topic_classifier.bias           | MISSING    | 
sentiment_classifier.bias       | MISSING    | 
topic_classifier.weight         | MISSING    | 
sentiment_classifier.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Multi-task model initialized successfully.


In [ ]:
class MultiTaskDataset(Dataset):
    def __init__(self, texts, sentiments, topics, tokenizer, max_len=256):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=max_len)
        self.sentiments = sentiments
        self.topics = topics

    def __len__(self):
        return len(self.sentiments)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels_sentiment'] = torch.tensor(self.sentiments[idx])
        item['labels_topic'] = torch.tensor(self.topics[idx])
        return item

mt_train_dataset = MultiTaskDataset(train_texts, train_labels, train_topics, tokenizer)
mt_dev_dataset = MultiTaskDataset(dev_texts, dev_labels, dev_topics, tokenizer)
mt_test_dataset = MultiTaskDataset(test_texts, test_labels, test_topics, tokenizer)

In [ ]:
def compute_mt_metrics(eval_pred):
    logits, labels = eval_pred
    # logits và labels ở đây sẽ là dictionary hoặc tuple tùy vào cách Trainer trả về
    # Do custom model trả về dict, ta cần xử lý logic tính metric riêng.
    # Để đơn giản, ta sẽ log loss trong quá trình train.
    return {} # Sẽ cập nhật chi tiết sau khi chạy thử

In [ ]:
mt_training_args = TrainingArguments(
    output_dir="./results_mt",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=False,
    remove_unused_columns=False
)

mt_trainer = Trainer(
    model=multi_task_model,
    args=mt_training_args,
    train_dataset=mt_train_dataset,
    eval_dataset=mt_dev_dataset
)

mt_trainer.train()

Epoch,Training Loss,Validation Loss
1,0.581236,0.576838
2,0.461250,0.526730
3,0.429714,0.545906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2145, training_loss=0.5534440260667067, metrics={'train_runtime': 1225.1119, 'train_samples_per_second': 27.979, 'train_steps_per_second': 1.751, 'total_flos': 2871386874270900.0, 'train_loss': 0.5534440260667067, 'epoch': 3.0})

In [ ]:
# Chạy dự đoán để tạo lại biến mt_predictions
mt_predictions = mt_trainer.predict(mt_test_dataset)
print("Đã tạo lại biến mt_predictions thành công.")

Đã tạo lại biến mt_predictions thành công.


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# 1. Trích xuất logits dựa trên kết quả debug (index 0 và 1)
logits_s = mt_predictions.predictions[0]
logits_t = mt_predictions.predictions[1]

preds_s = np.argmax(logits_s, axis=1)
preds_t = np.argmax(logits_t, axis=1)

# 2. In báo cáo kết quả
print("--- Sentiment Classification Report ---")
print(classification_report(test_labels, preds_s, target_names=['negative', 'neutral', 'positive'], digits=4))

print("\n--- Topic Classification Report ---")
print(classification_report(test_topics, preds_t, target_names=['lecturer', 'program', 'facility', 'others'], digits=4))

--- Sentiment Classification Report ---
              precision    recall  f1-score   support

    negative     0.9354    0.9560    0.9456      1409
     neutral     0.6373    0.3892    0.4833       167
    positive     0.9378    0.9579    0.9477      1590

    accuracy                         0.9270      3166
   macro avg     0.8368    0.7677    0.7922      3166
weighted avg     0.9209    0.9270    0.9223      3166


--- Topic Classification Report ---
              precision    recall  f1-score   support

    lecturer     0.9422    0.9393    0.9407      2290
     program     0.7484    0.8112    0.7785       572
    facility     0.9110    0.9172    0.9141       145
      others     0.6239    0.4591    0.5290       159

    accuracy                         0.8910      3166
   macro avg     0.8064    0.7817    0.7906      3166
weighted avg     0.8898    0.8910    0.8895      3166



In [ ]:
# Kiểm tra cấu trúc của predictions để debug lỗi IndexError
print(f"Type of predictions: {type(mt_predictions.predictions)}")
if isinstance(mt_predictions.predictions, (list, tuple)):
    print(f"Number of elements in tuple/list: {len(mt_predictions.predictions)}")
    for i, p in enumerate(mt_predictions.predictions):
        if hasattr(p, 'shape'):
            print(f"Element {i} shape: {p.shape}")
else:
    print(f"Predictions shape: {mt_predictions.predictions.shape}")

Type of predictions: <class 'tuple'>
Number of elements in tuple/list: 2
Element 0 shape: (3166, 3)
Element 1 shape: (3166, 4)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def predict_mt(text):
    multi_task_model.to(device)
    multi_task_model.eval()

    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = multi_task_model(**inputs)

    # Mapping labels
    sentiment_labels = ['negative', 'neutral', 'positive']
    topic_labels = ['lecturer', 'program', 'facility', 'others']

    idx_s = torch.argmax(outputs['logits_sentiment'], dim=1).item()
    idx_t = torch.argmax(outputs['logits_topic'], dim=1).item()

    print(f"Sentence: {text}")
    print(f"- Topic: {topic_labels[idx_t]}")
    print(f"- Sentiment: {sentiment_labels[idx_s]}")

# Thử nghiệm với ví dụ yêu cầu
predict_mt("giảng viên dạy rất hay")

Sentence: giảng viên dạy rất hay
- Topic: lecturer
- Sentiment: positive
